# PTB-XL ECG preprocessing for SDN experiments

This notebook regenerates the ECG input dataset used in the manuscript **"A Low-Power, Real-Time, and Noise-Resilient Spiking Neural Network for Bio-Signal Classification on a Neuromorphic Processor"**.

The ECG preprocessing steps were adapted from the PTB-XL experiment described by Wang et al. (2024), *Medformer: A multi-granularity patching transformer for medical time-series classification*. This refers only to the preprocessing steps, not to the full Medformer model architecture, training protocol, or evaluation setup. In particular, this notebook uses the high-resolution 500 Hz PTB-XL records, downsamples them to 250 Hz, applies standard scaling, and segments each recording into non-overlapping 1-second windows.

The saved SDN input shape is `(N, 12, 250)`, corresponding to `samples × channels × timestamps`.

**Note.** The SDN model performs 8-bit quantization and sigma-delta encoding internally. Those model operations are not part of this preprocessing notebook.


## 0. Expected raw PTB-XL directory structure

Download PTB-XL v1.0.3 from PhysioNet and arrange it as follows:

```text
PTB-XL/
├── ptbxl_database.csv
├── scp_statements.csv
└── records500/
    ├── 00000/
    │   ├── 00001_hr.dat
    │   ├── 00001_hr.hea
    │   └── ...
    └── ...
```

This notebook uses the high-resolution `records500` files through the `filename_hr` column in `ptbxl_database.csv`.

In [ ]:
from pathlib import Path
import ast
import json

import numpy as np
import pandas as pd
import wfdb
from scipy import interpolate
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
# ---------------------------------------------------------------------
# User configuration
# ---------------------------------------------------------------------
ROOT_DIR = Path('/path/to/PTB-XL')  # Change this to your local PTB-XL directory.
OUTPUT_DIR = Path('processed/ecg')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ORIGINAL_FS = 500
TARGET_FS = 250
SEGMENT_SECONDS = 1.0
SEGMENT_LENGTH = int(TARGET_FS * SEGMENT_SECONDS)  # 250 samples = 1 second at 250 Hz

# Manuscript-style stratified train/test split.
TEST_SIZE = 0.20
RANDOM_STATE = 42

# The adapted PTB-XL preprocessing steps include standard scaling after downsampling.
APPLY_STANDARD_SCALER = True

LABEL_TO_ID = {
    'NORM': 0,
    'MI': 1,
    'STTC': 2,
    'CD': 3,
    'HYP': 4,
}
ID_TO_LABEL = {v: k for k, v in LABEL_TO_ID.items()}


## 1. SCP-code to superclass mapping

Each ECG record contains a dictionary of SCP statements and likelihood scores. Following the original preprocessing script, the SCP statement with the highest likelihood is selected and mapped to one of the five PTB-XL diagnostic superclasses: `NORM`, `MI`, `STTC`, `CD`, and `HYP`.

In [ ]:
SCP_TO_SUPERCLASS = {
    # Normal ECG
    'NORM': 'NORM',

    # Myocardial infarction
    'IMI': 'MI', 'ASMI': 'MI', 'ILMI': 'MI', 'AMI': 'MI', 'ALMI': 'MI',
    'INJAS': 'MI', 'LMI': 'MI', 'INJAL': 'MI', 'IPLMI': 'MI', 'IPMI': 'MI',
    'INJIN': 'MI', 'INJLA': 'MI', 'PMI': 'MI', 'INJIL': 'MI',

    # ST/T change
    'NDT': 'STTC', 'NST_': 'STTC', 'DIG': 'STTC', 'ISC_': 'STTC',
    'ISCAL': 'STTC', 'LNGQT': 'STTC', 'ISCIN': 'STTC', 'ISCIL': 'STTC',
    'ISCAS': 'STTC', 'ISCLA': 'STTC', 'ANEUR': 'STTC', 'EL': 'STTC',
    'ISCAN': 'STTC',

    # Hypertrophy
    'LVH': 'HYP', 'LAO/LAE': 'HYP', 'RVH': 'HYP', 'RAO/RAE': 'HYP',
    'SEHYP': 'HYP',

    # Conduction disturbance
    'LAFB': 'CD', 'IRBBB': 'CD', '1AVB': 'CD', 'IVCD': 'CD',
    'CRBBB': 'CD', 'CLBBB': 'CD', 'LPFB': 'CD', 'WPW': 'CD',
    'ILBBB': 'CD', '3AVB': 'CD', '2AVB': 'CD',
}


def parse_scp_codes(raw_value):
    """Parse the PTB-XL scp_codes string into a dictionary."""
    if isinstance(raw_value, dict):
        return raw_value
    try:
        return ast.literal_eval(raw_value)
    except (ValueError, SyntaxError) as exc:
        raise ValueError(f'Could not parse scp_codes value: {raw_value}') from exc


def select_highest_probability_scp(raw_value):
    """Select the SCP code with the highest likelihood score."""
    scp_dict = parse_scp_codes(raw_value)
    if not scp_dict:
        return None
    return max(scp_dict, key=scp_dict.get)


def map_to_superclass(raw_value):
    """Map the highest-likelihood SCP code to a PTB-XL diagnostic superclass."""
    top_code = select_highest_probability_scp(raw_value)
    return SCP_TO_SUPERCLASS.get(top_code, 'others')

## 2. Load metadata and select valid records

Records are retained only if they map to one of the five target superclasses. Patients whose retained records map to inconsistent superclasses are excluded as part of the adapted preprocessing steps.


In [ ]:
database_path = ROOT_DIR / 'ptbxl_database.csv'
if not database_path.exists():
    raise FileNotFoundError(f'Cannot find {database_path}. Please update ROOT_DIR.')

metadata = pd.read_csv(database_path)
required_columns = {'ecg_id', 'patient_id', 'scp_codes', 'filename_hr'}
missing = required_columns.difference(metadata.columns)
if missing:
    raise ValueError(f'Missing required columns in ptbxl_database.csv: {sorted(missing)}')

metadata['top_scp_code'] = metadata['scp_codes'].apply(select_highest_probability_scp)
metadata['superclass'] = metadata['scp_codes'].apply(map_to_superclass)
metadata = metadata[metadata['superclass'] != 'others'].copy()
metadata['label_id'] = metadata['superclass'].map(LABEL_TO_ID).astype(int)

# Keep only patients with one consistent superclass across their retained records.
label_counts_per_patient = metadata.groupby('patient_id')['superclass'].nunique()
consistent_patient_ids = label_counts_per_patient[label_counts_per_patient == 1].index
selected_records = metadata[metadata['patient_id'].isin(consistent_patient_ids)].copy()
selected_records = selected_records.sort_values(['patient_id', 'ecg_id']).reset_index(drop=True)

# An anonymized contiguous subject key is used for release metadata.
subject_key_map = {pid: f'subject_{i:05d}' for i, pid in enumerate(sorted(selected_records['patient_id'].unique()), start=1)}
selected_records['subject_key'] = selected_records['patient_id'].map(subject_key_map)

print(f'Retained records: {len(selected_records):,}')
print(f'Retained subjects: {selected_records["patient_id"].nunique():,}')
print('Class distribution by record:')
print(selected_records['superclass'].value_counts().sort_index())

# Save only release-relevant metadata. The original PTB-XL patient_id is used internally
# for consistency filtering but is not included in the public release metadata.
release_columns = ['ecg_id', 'subject_key', 'filename_hr', 'top_scp_code', 'superclass', 'label_id']
selected_records[release_columns].to_csv(OUTPUT_DIR / 'ptbxl_selected_records.csv', index=False)

## 3. Signal processing functions

The PTB-XL signal preprocessing consists of:

- downsampling from 500 Hz to 250 Hz;
- record-wise standard scaling applied independently to each ECG lead;
- non-overlapping 1-second segmentation;
- saving each segment in `channels × timestamps` format, i.e., `(12, 250)`.

In [ ]:
def resample_linear(signal_2d, original_fs=ORIGINAL_FS, target_fs=TARGET_FS):
    """Resample a multi-channel signal by linear interpolation.

    Parameters
    ----------
    signal_2d : np.ndarray
        ECG signal with shape (samples, channels), as returned by wfdb.rdsamp.
    original_fs : int
        Original sampling frequency. PTB-XL records500 uses 500 Hz.
    target_fs : int
        Target sampling frequency. The manuscript uses 250 Hz.

    Returns
    -------
    np.ndarray
        Resampled signal with shape (new_samples, channels).
    """
    signal_2d = np.asarray(signal_2d, dtype=np.float32)
    n_samples, n_channels = signal_2d.shape
    n_target = int(round(n_samples * target_fs / original_fs))

    old_t = np.linspace(0, n_samples - 1, n_samples)
    new_t = np.linspace(0, n_samples - 1, n_target)

    channels = []
    for ch in range(n_channels):
        f = interpolate.interp1d(old_t, signal_2d[:, ch], kind='linear')
        channels.append(f(new_t))

    return np.stack(channels, axis=1).astype(np.float32)


def standardize_record(signal_2d):
    """Apply record-wise z-score normalization independently to each ECG lead.

    The scaler is fit on one ECG recording at a time. For each lead, the mean and
    standard deviation are computed over the temporal samples of that recording.
    """
    scaler = StandardScaler()
    return scaler.fit_transform(signal_2d).astype(np.float32)


def segment_nonoverlap(signal_2d, segment_length=SEGMENT_LENGTH):
    """Split a signal into non-overlapping fixed-length windows.

    Parameters
    ----------
    signal_2d : np.ndarray
        Signal with shape (samples, channels).
    segment_length : int
        Number of samples per segment. At 250 Hz, 250 samples correspond to 1 second.

    Returns
    -------
    np.ndarray
        Segments with shape (num_segments, channels, segment_length).
    """
    n_samples, n_channels = signal_2d.shape
    n_segments = n_samples // segment_length
    if n_segments == 0:
        return np.empty((0, n_channels, segment_length), dtype=np.float32)

    trimmed = signal_2d[:n_segments * segment_length]
    segments = trimmed.reshape(n_segments, segment_length, n_channels)
    return np.transpose(segments, (0, 2, 1)).astype(np.float32)

## 4. Generate ECG segments

This cell loads each selected 500 Hz ECG record, resamples it to 250 Hz, applies record-wise standard scaling, and segments it into non-overlapping 1-second windows.

In [ ]:
all_segments = []
all_labels = []
segment_metadata_rows = []

for _, row in selected_records.iterrows():
    record_path = ROOT_DIR / row['filename_hr']

    # wfdb.rdsamp expects the path without the .hea/.dat extension.
    ecg_signal, fields = wfdb.rdsamp(str(record_path))

    if ecg_signal.shape[1] != 12:
        raise ValueError(f'Expected 12 leads for ecg_id={row.ecg_id}, got shape {ecg_signal.shape}.')

    resampled = resample_linear(ecg_signal, ORIGINAL_FS, TARGET_FS)

    if APPLY_STANDARD_SCALER:
        processed = standardize_record(resampled)
    else:
        processed = resampled

    segments = segment_nonoverlap(processed, SEGMENT_LENGTH)
    label_id = int(row['label_id'])
    label_name = row['superclass']

    all_segments.append(segments)
    all_labels.extend([label_id] * len(segments))

    for seg_idx in range(len(segments)):
        segment_metadata_rows.append({
            'global_segment_index': len(segment_metadata_rows),
            'subject_key': row['subject_key'],
            'ecg_id': row['ecg_id'],
            'filename_hr': row['filename_hr'],
            'segment_index_within_record': seg_idx,
            'start_sample_250hz': seg_idx * SEGMENT_LENGTH,
            'end_sample_250hz': (seg_idx + 1) * SEGMENT_LENGTH,
            'label_id': label_id,
            'label_name': label_name,
        })

if not all_segments:
    raise RuntimeError('No ECG segments were generated. Check ROOT_DIR and preprocessing filters.')

X = np.concatenate(all_segments, axis=0).astype(np.float32)
y = np.asarray(all_labels, dtype=np.int64)
segment_metadata = pd.DataFrame(segment_metadata_rows)

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print('Segment class distribution:')
print(pd.Series(y).map(ID_TO_LABEL).value_counts().sort_index())

## 5. Split and save

The generated segments are saved using a manuscript-style stratified segment-level train/test split. This split is part of the released dataset-generation code for the present study and should not be interpreted as adopting the full Medformer experimental setup. It is also not a subject-independent split; segments from the same patient or ECG record may appear in both train and test subsets.


In [ ]:
indices = np.arange(len(y))
segment_metadata['split'] = 'unused'

train_index, test_index = train_test_split(
    indices,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_train = X[train_index]
X_test = X[test_index]
y_train = y[train_index]
y_test = y[test_index]

segment_metadata.loc[train_index, 'split'] = 'train'
segment_metadata.loc[test_index, 'split'] = 'test'

npz_path = OUTPUT_DIR / 'ptbxl_250hz_standardized_1s_train_test.npz'
np.savez_compressed(
    npz_path,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    train_index=train_index,
    test_index=test_index,
)

print(f'Saved: {npz_path}')
print(f'Train shape: {X_train.shape}, Test shape: {X_test.shape}')
print('Train class distribution:')
print(pd.Series(y_train).map(ID_TO_LABEL).value_counts().sort_index())
print('Test class distribution:')
print(pd.Series(y_test).map(ID_TO_LABEL).value_counts().sort_index())

segment_metadata.to_csv(OUTPUT_DIR / 'ptbxl_segment_metadata.csv', index=False)

with open(OUTPUT_DIR / 'ptbxl_label_mapping.json', 'w', encoding='utf-8') as f:
    json.dump({'label_to_id': LABEL_TO_ID, 'id_to_label': ID_TO_LABEL}, f, indent=2)
